# JEPA-Coder — Phase 1: Pretraining

**Objective:** Next-token prediction on The Stack v2 (Python) + C4, producing `pretrained_reasoner.pt` for Phase 2 SST.

**Spec:** `docs/jepa_coder_architecture_v2.md §4 Phase 1`

| Hyperparameter | Value |
|---|---|
| Architecture | Reasoner 768d, 16L, 12H, 3072 FFN |
| Context | 1024 tokens |
| Eff. batch | 128 (32 seqs × 4 accum steps) |
| LR | 3e-4, cosine decay, 2000-step warmup |
| Steps | 100K |
| Precision | BF16 |
| Target GPU | Colab L4 / A100 |
| Est. runtime | ~55–140 h on L4, ~28–70 h on A100 |

## 1. Setup

Clones the repo, installs requirements, and mounts Google Drive for checkpoint storage.

> **Before running:** switch the runtime to **L4** or **A100** (`Runtime → Change runtime type → GPU`).

In [ ]:
import os, sys

# ── Clone / update repo ──────────────────────────────────────────────────────
REPO_ROOT = '/content/jepa-coder'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/pu-suo/jepa-coder.git {REPO_ROOT}
else:
    !cd {REPO_ROOT} && git pull

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# ── Install requirements ─────────────────────────────────────────────────────
%pip install -q -r {REPO_ROOT}/requirements.txt

# ── Mount Drive and set checkpoint directory ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/jepa_coder_data/checkpoints/pretrain'
os.makedirs(CKPT_DIR, exist_ok=True)

print(f'Repo root  : {REPO_ROOT}')
print(f'Checkpoints: {CKPT_DIR}')

## 2. Hugging Face Authentication

`bigcode/the-stack-v2-dedup` is gated. Accept the licence at [huggingface.co/datasets/bigcode/the-stack-v2-dedup](https://huggingface.co/datasets/bigcode/the-stack-v2-dedup) then paste a read token below.

In [ ]:
import getpass, os
from huggingface_hub import login

hf_token = getpass.getpass('HuggingFace read token: ')
os.environ['HF_TOKEN'] = hf_token          # ensures all HF code paths find the token
login(token=hf_token, add_to_git_credential=False)
print('Logged in to HuggingFace.')

## 3. Configure the run

In [ ]:
import dataclasses
from training.pretrain import PretrainConfig

config = PretrainConfig(
    # ── Model (must match spec §2.1) ────────────────────────────────────────
    dim            = 768,
    n_layers       = 16,
    n_heads        = 12,
    ffn_dim        = 3072,
    context_length = 1024,

    # ── Optimizer (spec §4 Phase 1) ─────────────────────────────────────────
    lr             = 3e-4,
    weight_decay   = 0.1,
    warmup_steps   = 2000,
    max_steps      = 100_000,

    # ── Batching: eff. batch = seqs_per_accum × accum_steps = 128 ───────────
    seqs_per_accum = 32,
    accum_steps    = 4,

    # ── Data ────────────────────────────────────────────────────────────────
    stack_language_filter = 'Python',
    c4_mix_ratio          = 0.2,
    data_seed             = 42,

    # ── Checkpointing → Drive ───────────────────────────────────────────────
    checkpoint_dir   = CKPT_DIR,
    checkpoint_every = 2000,

    # ── Logging ─────────────────────────────────────────────────────────────
    log_every = 10,
)

print('Config:')
for k, v in dataclasses.asdict(config).items():
    print(f'  {k}: {v}')

## 4. Inspect existing checkpoints

Lists what is already on Drive. `latest.pt` is the resume target; numbered `.pt` files are milestone saves.

In [ ]:
import os, torch

ckpt_files = sorted(
    f for f in os.listdir(CKPT_DIR) if f.endswith('.pt')
) if os.path.isdir(CKPT_DIR) else []

if ckpt_files:
    print(f'{len(ckpt_files)} checkpoint(s) in {CKPT_DIR}:')
    for f in ckpt_files:
        path = os.path.join(CKPT_DIR, f)
        print(f'  {f}  ({os.path.getsize(path) / 1e6:.0f} MB)')

    if 'latest.pt' in ckpt_files:
        meta = torch.load(os.path.join(CKPT_DIR, 'latest.pt'),
                          map_location='cpu', weights_only=True)
        print(f"\nlatest.pt → step {meta['step']:,}  "
              f"({meta['total_tokens'] / 1e9:.2f}B tokens seen)")
else:
    print('No checkpoints found — will start from scratch.')

## 5. Data pre-flight check

Fetches one token chunk from the live data pipeline before committing to a 100K-step run. Confirms authentication and streaming are working. Expected output: `✓ Data pipeline OK` within ~2 min.

In [ ]:
import gc
from transformers import AutoTokenizer
from training.pretrain import build_data_iterator

print('Pre-flight data check — fetching first chunk (first shard download may take 1–2 min)...')
_tok = AutoTokenizer.from_pretrained('bigcode/starcoder2-3b')
_iter = build_data_iterator(config, _tok)
_chunk = next(_iter)
print(f'✓ Data pipeline OK  shape={tuple(_chunk.shape)}  '
      f'first tokens={_chunk[:8].tolist()}')
del _tok, _iter, _chunk; gc.collect()

## 6. Run pretraining

`pretrain()` resumes automatically from `latest.pt` if one exists. A tqdm bar shows live progress; the bar postfix updates every `log_every` steps with current loss, LR, and tokens seen. Progress is also appended to `training_logs.csv` in `CKPT_DIR`.

In [ ]:
from training.pretrain import pretrain

pretrain(config)

## 7. Verify the final checkpoint

In [ ]:
import os, torch

final_ckpt = os.path.join(CKPT_DIR, 'pretrained_reasoner.pt')
if not os.path.isfile(final_ckpt):
    final_ckpt = os.path.join(CKPT_DIR, 'latest.pt')
    print('pretrained_reasoner.pt not found — loading latest.pt instead')

ckpt = torch.load(final_ckpt, map_location='cpu', weights_only=True)
n_params = sum(t.numel() for t in ckpt['model_state_dict'].values())

print(f'File        : {final_ckpt}')
print(f'Step        : {ckpt["step"]:,}')
print(f'Tokens seen : {ckpt["total_tokens"] / 1e9:.3f}B')
print(f'Model params: {n_params / 1e6:.1f}M')

## 8. Plot training curves

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

log_path = os.path.join(CKPT_DIR, 'training_logs.csv')
df = pd.read_csv(log_path)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(df['step'], df['loss'], linewidth=0.8)
ax1.set_title('Training Loss vs. Step')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.grid(True, alpha=0.3)

ax2.plot(df['step'], df['lr'], linewidth=0.8, color='tab:orange')
ax2.set_title('Learning Rate vs. Step')
ax2.set_xlabel('Step')
ax2.set_ylabel('LR')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Rows logged : {len(df):,}')
print(f'Final loss  : {df["loss"].iloc[-1]:.4f}')
print(f'Final step  : {df["step"].iloc[-1]:,}')